In [3]:

# SWYNEX Task 2 - Machine Learning Model

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load Titanic dataset
import seaborn as sns
df = sns.load_dataset("titanic")

# Select features and target
df = df[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "survived"]]
df = df.dropna(subset=["survived"])

X = df.drop("survived", axis=1)
y = df["survived"]

# Separate numerical and categorical columns
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["pclass", "sex", "embarked"]

# Preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model 1: Logistic Regression
model1 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

model1.fit(X_train, y_train)
pred1 = model1.predict(X_test)

# Model 2: Decision Tree
model2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
])

model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)

# Evaluate both models
acc1 = accuracy_score(y_test, pred1)
acc2 = accuracy_score(y_test, pred2)

print("===== MODEL EVALUATION =====")
print("Logistic Regression Accuracy:", round(acc1 * 100, 2), "%")
print("Decision Tree Accuracy:", round(acc2 * 100, 2), "%")

print("\n===== LOGISTIC REGRESSION REPORT =====")
print(classification_report(y_test, pred1))

print("\n===== DECISION TREE REPORT =====")
print(classification_report(y_test, pred2))

# Comparison
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy (%)": [round(acc1 * 100, 2), round(acc2 * 100, 2)]
})

print("\n===== MODEL COMPARISON =====")
display(results)

best_model = "Logistic Regression" if acc1 >= acc2 else "Decision Tree"
print("\nBetter performing model:", best_model)


===== MODEL EVALUATION =====
Logistic Regression Accuracy: 80.45 %
Decision Tree Accuracy: 75.98 %

===== LOGISTIC REGRESSION REPORT =====
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


===== DECISION TREE REPORT =====
              precision    recall  f1-score   support

           0       0.77      0.87      0.82       110
           1       0.74      0.58      0.65        69

    accuracy                           0.76       179
   macro avg       0.75      0.73      0.73       179
weighted avg       0.76      0.76      0.75       179


===== MODEL COMPARISON =====


,Model,Accuracy (%)
0,Logistic Regression,80.45
1,Decision Tree,75.98



Better performing model: Logistic Regression


In [4]:

# SWYNEX Task 3 - Model Improvement
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning for Logistic Regression
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

grid_search = GridSearchCV(
    model1,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_
best_predictions = best_model.predict(X_test)

# Compare original and improved models
original_accuracy = accuracy_score(y_test, pred1)
improved_accuracy = accuracy_score(y_test, best_predictions)

print("===== MODEL IMPROVEMENT RESULTS =====")
print("Original Logistic Regression:", round(original_accuracy * 100, 2), "%")
print("Improved Logistic Regression:", round(improved_accuracy * 100, 2), "%")
print("Best Parameters:", grid_search.best_params_)
print("Cross-Validation Score:", round(grid_search.best_score_ * 100, 2), "%")

print("\n===== IMPROVED CLASSIFICATION REPORT =====")
print(classification_report(y_test, best_predictions))

comparison = pd.DataFrame({
    "Model": ["Original Logistic Regression", "Tuned Logistic Regression"],
    "Accuracy (%)": [
        round(original_accuracy * 100, 2),
        round(improved_accuracy * 100, 2)
    ]
})

display(comparison)


===== MODEL IMPROVEMENT RESULTS =====
Original Logistic Regression: 80.45 %
Improved Logistic Regression: 81.01 %
Best Parameters: {'model__C': 0.1, 'model__solver': 'liblinear'}
Cross-Validation Score: 79.78 %

===== IMPROVED CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       110
           1       0.81      0.67      0.73        69

    accuracy                           0.81       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.81      0.81      0.81       179



,Model,Accuracy (%)
0,Original Logistic Regression,80.45
1,Tuned Logistic Regression,81.01


In [5]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

grid_search = GridSearchCV(
    model1,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_predictions = best_model.predict(X_test)

original_accuracy = accuracy_score(y_test, pred1)
improved_accuracy = accuracy_score(y_test, best_predictions)

print("===== TASK 3 MODEL IMPROVEMENT =====")
print("Original Accuracy:", round(original_accuracy * 100, 2), "%")
print("Improved Accuracy:", round(improved_accuracy * 100, 2), "%")
print("Best Parameters:", grid_search.best_params_)
print("Cross-Validation Score:", round(grid_search.best_score_ * 100, 2), "%")

comparison = pd.DataFrame({
    "Model": ["Original Logistic Regression", "Tuned Logistic Regression"],
    "Accuracy (%)": [
        round(original_accuracy * 100, 2),
        round(improved_accuracy * 100, 2)
    ]
})

display(comparison)

print("\n===== IMPROVED CLASSIFICATION REPORT =====")
print(classification_report(y_test, best_predictions))


===== TASK 3 MODEL IMPROVEMENT =====
Original Accuracy: 80.45 %
Improved Accuracy: 81.01 %
Best Parameters: {'model__C': 0.1, 'model__solver': 'liblinear'}
Cross-Validation Score: 79.78 %


,Model,Accuracy (%)
0,Original Logistic Regression,80.45
1,Tuned Logistic Regression,81.01



===== IMPROVED CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       110
           1       0.81      0.67      0.73        69

    accuracy                           0.81       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.81      0.81      0.81       179

